In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "password"
MINIO_ENDPOINT = "http://minio:9000"

PACKAGES = [
    "io.delta:delta-spark_2.12:3.0.0", 
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
    "org.apache.hadoop:hadoop-aws:3.3.4"
]

spark = SparkSession.builder \
    .appName("FinStream-Ingestion") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session has started!")

Spark Session has started!


In [11]:
# The schema of the incoming JSON data
trade_schema = StructType([
    StructField("trade_id", StringType()),
    StructField("symbol", StringType()),
    StructField("price", DoubleType()),
    StructField("quantity", DoubleType()),
    StructField("side", StringType()),
    StructField("timestamp", StringType()) # Önce String okuyup sonra Cast edeceğiz
])

# read kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "financial_trades") \
    .option("startingOffsets", "earliest") \
    .load()

# JSON Parse and clean
df_trades = df_kafka.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), trade_schema).alias("data")) \
    .select("data.*") \
    .withColumn("timestamp", col("timestamp").cast(TimestampType()))

# print and test (Console Sink)
# query = df_trades.writeStream.format("console").start()
# query.stop()

In [9]:
# Write the data to the ‘lakehouse’ bucket in MinIO
# Checkpoint: So Spark doesn't forget where it left off
CHECKPOINT_LOC = "s3a://lakehouse/checkpoints/trades"
DELTA_TABLE_PATH = "s3a://lakehouse/bronze/trades"

query = df_trades.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_LOC) \
    .start(DELTA_TABLE_PATH)

print(f"Streaming has started... data is writing here: {DELTA_TABLE_PATH}")
# query.awaitTermination()

Streaming has started... data is writing here: s3a://lakehouse/bronze/trades
